# LeetCode 893 — Groups of Special-Equivalent Strings bằng Python

Link đề: https://leetcode.com/problems/groups-of-special-equivalent-strings/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), String (chuỗi), Sorting (sắp xếp)

Skeleton LeetCode:

```python
class Solution:
    def numSpecialEquivGroups(self, words: List[str]) -> int:
        ...
```

Ghi chú: mỗi nước đi **hoán đổi hai ký tự ở hai vị trí chẵn** hoặc **hai vị trí lẻ** của một chuỗi. Vì có thể thực hiện bao nhiêu nước tùy ý, các ký tự ở vị trí chẵn có thể được **sắp xếp lại tùy ý** với nhau, các ký tự ở vị trí lẻ cũng vậy, nhưng **không bao giờ đổi chỗ giữa hai nhóm chẵn và lẻ**. Do đó hai chuỗi tương đương đặc biệt khi và chỉ khi có **cùng đa tập ký tự ở vị trí chẵn** và **cùng đa tập ký tự ở vị trí lẻ**: đó chính là "dấu vân tay" để gom nhóm.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai chuỗi cùng độ dài là **tương đương đặc biệt** nếu biến được chuỗi này thành chuỗi kia bằng các phép hoán đổi hai ký tự chỉ số chẵn hoặc hai ký tự chỉ số lẻ; đếm số nhóm tương đương lớn nhất.
2. Nắm các công cụ Python cần thiết: cắt lát có bước `s[::2]` và `s[1::2]`, `sorted` và `"".join`, `set` để đếm khóa khác nhau, `collections.Counter`, `itertools.combinations` để liệt kê cặp chỉ số, và bảng đếm 26 chữ cái.
3. Hiểu nguyên lý: quan hệ "biến đổi được" là **quan hệ tương đương** nên đếm nhóm bằng cách gán mỗi chuỗi một **dạng chuẩn** (canonical form) rồi đếm số dạng chuẩn khác nhau.
4. Đi qua 4 cách: BFS toàn bộ quỹ đạo các chuỗi đạt được (brute, đúng định nghĩa), so sánh từng cặp với đại diện nhóm, khóa là chuỗi đã sắp xếp theo từng nhóm chẵn/lẻ (cách nộp chính), và khóa là bảng đếm chữ cái; đối chiếu bằng hai oracle độc lập (BFS quỹ đạo và thuật toán hoán đổi cấu trúc).
5. Tránh các lỗi kinh điển: coi như anagram (sắp xếp cả chuỗi), chỉ xét nhóm chẵn hoặc chỉ nhóm lẻ, dùng `set` thay cho đa tập, và gộp nhầm hai chuỗi có nhóm chẵn/lẻ bị hoán đổi vai trò.

## 2. Đề bài & ví dụ

Cho mảng các chuỗi **cùng độ dài** `words`.

Mỗi **nước đi**, ta có thể **hoán đổi hai ký tự ở hai chỉ số chẵn bất kỳ** hoặc **hai ký tự ở hai chỉ số lẻ bất kỳ** của một chuỗi `words[i]`.

Hai chuỗi `words[i]` và `words[j]` là **tương đương đặc biệt** nếu sau một số nước đi bất kỳ (kể cả không đi) ta có `words[i] == words[j]`.

- Ví dụ: `"zzxy"` và `"xyzz"` tương đương đặc biệt vì `"zzxy" -> "xzzy" -> "xyzz"`.

Một **nhóm các chuỗi tương đương đặc biệt** là một tập con không rỗng của `words` sao cho (1) mọi cặp trong nhóm tương đương đặc biệt và (2) nhóm có kích thước lớn nhất có thể.

Trả về **số nhóm** các chuỗi tương đương đặc biệt trong `words`.

**Ví dụ 1**

- Input: `words = ["abcd","cdab","cbad","xyzz","zzxy","zzyx"]`
- Output: `3`
- Giải thích: nhóm `["abcd","cdab","cbad"]`, nhóm `["xyzz","zzxy"]`, nhóm `["zzyx"]`; chú ý `"zzxy"` không tương đương `"zzyx"`.

**Ví dụ 2**

- Input: `words = ["abc","acb","bac","bca","cab","cba"]`
- Output: `3`

**Ràng buộc:**

- `1 <= words.length <= 1000`
- `1 <= words[i].length <= 20`
- `words[i]` gồm chữ cái tiếng Anh thường.
- Mọi `words[i]` có cùng độ dài.

In [1]:
from typing import List
from itertools import combinations
from collections import Counter
from math import factorial
import random, string, time

examples = [
    (["abcd", "cdab", "cbad", "xyzz", "zzxy", "zzyx"], 3),
    (["abc", "acb", "bac", "bca", "cab", "cba"], 3),
]
for words, expected in examples:
    print(f"words={words} kỳ vọng={expected}")

words=['abcd', 'cdab', 'cbad', 'xyzz', 'zzxy', 'zzyx'] kỳ vọng=3
words=['abc', 'acb', 'bac', 'bca', 'cab', 'cba'] kỳ vọng=3


## 3. Kiến thức nền

### 3.1. Cắt lát có bước: chỉ số chẵn và chỉ số lẻ

`s[::2]` lấy các ký tự ở chỉ số `0, 2, 4, ...` (chẵn) và `s[1::2]` lấy ở chỉ số `1, 3, 5, ...` (lẻ).

In [2]:
s = "abcdefg"
print("chẵn:", s[::2], "| lẻ:", s[1::2])
s = "zzxy"
print("chẵn:", s[::2], "| lẻ:", s[1::2])

chẵn: aceg | lẻ: bdf
chẵn: zx | lẻ: zy


### 3.2. Hoán đổi cùng chẵn/lẻ không đưa ký tự sang nhóm khác

Một nước đi hoán đổi `s[i]` và `s[j]` với `i`, `j` **cùng tính chẵn lẻ**. Vì vậy ký tự ở vị trí chẵn **luôn ở vị trí chẵn** và ký tự ở vị trí lẻ luôn ở vị trí lẻ: đa tập ký tự của nhóm chẵn và đa tập của nhóm lẻ là **bất biến**.

In [3]:
def apply_swap(s: str, i: int, j: int) -> str:
    assert i % 2 == j % 2, "chỉ được hoán đổi hai chỉ số cùng chẵn hoặc cùng lẻ"
    t = list(s)
    t[i], t[j] = t[j], t[i]
    return "".join(t)

w = "zzxy"
w1 = apply_swap(w, 0, 2)       # zzxy -> xzzy
w2 = apply_swap(w1, 1, 3)      # xzzy -> xyzz
print(w, "->", w1, "->", w2)
for u in (w, w1, w2):
    print(u, "| đa tập chẵn", sorted(u[::2]), "| đa tập lẻ", sorted(u[1::2]))

zzxy -> xzzy -> xyzz
zzxy | đa tập chẵn ['x', 'z'] | đa tập lẻ ['y', 'z']
xzzy | đa tập chẵn ['x', 'z'] | đa tập lẻ ['y', 'z']
xyzz | đa tập chẵn ['x', 'z'] | đa tập lẻ ['y', 'z']


### 3.3. Dạng chuẩn (canonical form) và đếm nhóm

Quan hệ tương đương đặc biệt có tính **phản xạ, đối xứng và bắc cầu** (hoán đổi là khả nghịch, và các nước đi nối tiếp nhau), nên các chuỗi chia thành các **lớp** rời nhau. Ta gán mỗi chuỗi một **khóa** `key(s)` sao cho `key(a) == key(b)` khi và chỉ khi `a`, `b` tương đương; khi đó số nhóm bằng số khóa khác nhau: `len({key(w) for w in words})`.

In [4]:
def key_sorted(s: str) -> str:
    return "".join(sorted(s[::2]) + sorted(s[1::2]))

for w in ("zzxy", "xyzz", "zzyx", "abcd", "cdab", "cbad"):
    print(f"{w}: khóa {key_sorted(w)}")

zzxy: khóa xzyz
xyzz: khóa xzyz
zzyx: khóa yzxz
abcd: khóa acbd
cdab: khóa acbd
cbad: khóa acbd


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: tách mỗi chuỗi thành nhóm chẵn và nhóm lẻ rồi sắp xếp từng nhóm.

| Chuỗi | Chẵn (sắp xếp) | Lẻ (sắp xếp) |
|---|---|---|
| `abcd` | `ac` | `bd` |
| `cdab` | `ac` | `bd` |
| `cbad` | `ca` -> `ac` | `bd` |
| `xyzz` | `xz` | `yz` |
| `zzxy` | `xz` | `yz` |
| `zzyx` | `yz` | `xz` |

Ba nhóm khóa: `(ac, bd)`, `(xz, yz)`, `(yz, xz)`. Chú ý `zzxy` và `zzyx` có **hai nhóm bị đảo vai trò** (chẵn của chuỗi này là lẻ của chuỗi kia) nên **không** tương đương: đáp án `3`.

## 5. Cách 1 — Brute Force: BFS toàn bộ quỹ đạo

**Ý tưởng.** Đúng định nghĩa: từ một chuỗi `w`, liệt kê **mọi chuỗi đạt được** bằng cách thực hiện hoán đổi cùng chẵn/lẻ lặp đi lặp lại (BFS/DFS trên các chuỗi đã gặp). Hai chuỗi tương đương khi chuỗi này thuộc quỹ đạo của chuỗi kia. Đếm nhóm: duyệt từng từ, nếu nó thuộc quỹ đạo của một **đại diện** đã có thì cùng nhóm, ngược lại tạo nhóm mới (và tính quỹ đạo của nó).

Quỹ đạo có thể rất lớn (`(10!)^2` với `L = 20`), nên cách này chỉ dùng được cho `L` nhỏ; nó là **oracle theo định nghĩa**.

In [5]:
def orbit(w: str) -> set:
    seen = {w}
    stack = [w]
    L = len(w)
    while stack:
        s = stack.pop()
        for parity in (0, 1):
            for i, j in combinations(range(parity, L, 2), 2):
                t = list(s)
                t[i], t[j] = t[j], t[i]
                t = "".join(t)
                if t not in seen:
                    seen.add(t)
                    stack.append(t)
    return seen

def groups_brute(words: List[str]) -> int:
    reps: List[set] = []                       # quỹ đạo của từng đại diện
    for w in words:
        if not any(w in o for o in reps):
            reps.append(orbit(w))
    return len(reps)

for words, expected in examples:
    got = groups_brute(words)
    print(words, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['abcd', 'cdab', 'cbad', 'xyzz', 'zzxy', 'zzyx'] -> 3 OK
['abc', 'acb', 'bac', 'bca', 'cab', 'cba'] -> 3 OK


**Độ phức tạp và nút thắt.** Quỹ đạo của một chuỗi độ dài `L` có tối đa `ceil(L/2)! * floor(L/2)!` phần tử: bùng nổ giai thừa. Xem kích thước tăng:

In [6]:
for L in (4, 6, 8, 9, 10):
    w = string.ascii_lowercase[:L]                   # mọi ký tự khác nhau: quỹ đạo lớn nhất
    t0 = time.perf_counter()
    size = len(orbit(w))
    print(f"L={L:2}: quỹ đạo {size:>9} chuỗi, {1000 * (time.perf_counter() - t0):9.1f} ms")

L= 4: quỹ đạo         4 chuỗi,       0.0 ms
L= 6: quỹ đạo        36 chuỗi,       0.1 ms
L= 8: quỹ đạo       576 chuỗi,       2.1 ms
L= 9: quỹ đạo      2880 chuỗi,      16.7 ms
L=10: quỹ đạo     14400 chuỗi,      85.8 ms


Quỹ đạo tăng theo giai thừa. Nút thắt: ta liệt kê cả **trạng thái trung gian** trong khi chỉ cần biết hai chuỗi có cùng **đa tập** theo chẵn/lẻ hay không.

## 6. Ý tưởng thuật toán / bước đột phá

**Các nước đi cho phép sắp xếp lại tùy ý trong mỗi nhóm chẵn/lẻ.** Hoán đổi bất kỳ hai chỉ số cùng chẵn/lẻ sinh ra **mọi hoán vị** của nhóm đó (mỗi hoán vị là tích các phép đổi chỗ). Do đó chuỗi `a` biến được thành chuỗi `b` khi và chỉ khi:

- đa tập ký tự chỉ số chẵn của `a` bằng của `b`, **và**
- đa tập ký tự chỉ số lẻ của `a` bằng của `b`.

Hai điều kiện này chỉ cần so sánh **dạng đã sắp xếp** (hoặc bảng đếm) của từng nhóm. Vậy ta có thể gán mỗi chuỗi một khóa và đếm số khóa khác nhau.

| Cách | Ý tưởng |
|---|---|
| 2 | So sánh từng cặp: với mỗi từ, kiểm tra có tương đương với đại diện nhóm nào đó không (dạng đã sắp xếp) |
| 3 | Khóa = `sorted(chẵn) + sorted(lẻ)`; đếm `len(set)` (nộp) |
| 4 | Khóa = bảng đếm 26 chữ cái cho chẵn và cho lẻ (không sắp xếp) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — So sánh với đại diện nhóm

**Giải thích từng đoạn.**

- `reps`: danh sách đại diện của các nhóm đã tìm thấy.
- Với mỗi từ `w`: kiểm tra `equivalent(w, r)` cho từng đại diện `r`; nếu có thì cùng nhóm, nếu không thì `w` trở thành đại diện mới.
- `equivalent(a, b)`: `sorted(a[::2]) == sorted(b[::2]) and sorted(a[1::2]) == sorted(b[1::2])`.
- Quan hệ bắc cầu nên chỉ cần so với **một** đại diện mỗi nhóm.

Chi phí `O(w * g * L log L)` với `g` là số nhóm (tệ nhất `g = w`): chậm hơn Cách 3 nhưng không cần hiểu khóa.

In [7]:
def equivalent(a: str, b: str) -> bool:
    return sorted(a[::2]) == sorted(b[::2]) and sorted(a[1::2]) == sorted(b[1::2])

def groups_reps(words: List[str]) -> int:
    reps: List[str] = []
    for w in words:
        if not any(equivalent(w, r) for r in reps):
            reps.append(w)
    return len(reps)

for words, expected in examples:
    got = groups_reps(words)
    print(words, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['abcd', 'cdab', 'cbad', 'xyzz', 'zzxy', 'zzyx'] -> 3 OK
['abc', 'acb', 'bac', 'bca', 'cab', 'cba'] -> 3 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def numSpecialEquivGroups(self, words: List[str]) -> int:
        reps: List[str] = []
        for w in words:
            if not any(sorted(w[::2]) == sorted(r[::2]) and sorted(w[1::2]) == sorted(r[1::2]) for r in reps):
                reps.append(w)
        return len(reps)

sol_reps = Solution().numSpecialEquivGroups
assert [sol_reps(w) for w, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(w * g * L log L)` với `g` là số nhóm (tệ nhất `w`); bộ nhớ `O(g * L)`.

---

### 7.2. Cách 3 — Khóa là chuỗi đã sắp xếp theo chẵn/lẻ (cách nộp chính)

**Ý tưởng khác.** Với mỗi chuỗi tính **khóa** `"".join(sorted(s[::2]) + sorted(s[1::2]))`: các ký tự chẵn đã sắp xếp nối với các ký tự lẻ đã sắp xếp. Vì mọi chuỗi cùng độ dài nên số ký tự chẵn cố định, phép nối **không gây nhập nhằng** ranh giới giữa nhóm chẵn và lẻ. Đáp án là số khóa khác nhau (dùng `set`).

**Giải thích từng đoạn.**

- `sorted(s[::2])`: danh sách ký tự chẵn đã sắp xếp.
- `+ sorted(s[1::2])`: nối thêm ký tự lẻ đã sắp xếp (nối hai danh sách).
- `"".join(...)`: thành chuỗi để băm.
- `len({key(w) for w in words})`.

In [9]:
def groups_sorted(words: List[str]) -> int:
    return len({"".join(sorted(w[::2]) + sorted(w[1::2])) for w in words})

for words, expected in examples:
    got = groups_sorted(words)
    print(words, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['abcd', 'cdab', 'cbad', 'xyzz', 'zzxy', 'zzyx'] -> 3 OK
['abc', 'acb', 'bac', 'bca', 'cab', 'cba'] -> 3 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def numSpecialEquivGroups(self, words: List[str]) -> int:
        return len({"".join(sorted(w[::2]) + sorted(w[1::2])) for w in words})

sol_sorted = Solution().numSpecialEquivGroups
assert [sol_sorted(w) for w, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(w * L log L)` thời gian (sắp xếp hai nửa mỗi chuỗi); bộ nhớ `O(w * L)` cho tập khóa.

---

### 7.3. Cách 4 — Khóa là bảng đếm chữ cái (không sắp xếp)

**Ý tưởng khác.** Thay sắp xếp bằng **đếm**: tạo mảng `cnt` gồm `52` phần tử: `cnt[c]` là số lần chữ `c` xuất hiện ở vị trí chẵn, `cnt[26 + c]` là số lần xuất hiện ở vị trí lẻ. Dùng `tuple(cnt)` làm khóa. Thời gian `O(L)` mỗi chuỗi.

**Giải thích từng đoạn.** `ord(ch) - 97` là chỉ số chữ cái; `i & 1` cho `0` (chẵn) hoặc `1` (lẻ), nên `cnt[(i & 1) * 26 + ord(ch) - 97] += 1`.

In [11]:
def groups_count(words: List[str]) -> int:
    seen = set()
    for w in words:
        cnt = [0] * 52
        for i, ch in enumerate(w):
            cnt[(i & 1) * 26 + ord(ch) - 97] += 1
        seen.add(tuple(cnt))
    return len(seen)

for words, expected in examples:
    got = groups_count(words)
    print(words, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['abcd', 'cdab', 'cbad', 'xyzz', 'zzxy', 'zzyx'] -> 3 OK
['abc', 'acb', 'bac', 'bca', 'cab', 'cba'] -> 3 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def numSpecialEquivGroups(self, words: List[str]) -> int:
        seen = set()
        for w in words:
            cnt = [0] * 52
            for i, ch in enumerate(w):
                cnt[(i & 1) * 26 + ord(ch) - 97] += 1
            seen.add(tuple(cnt))
        return len(seen)

sol_count = Solution().numSpecialEquivGroups
assert [sol_count(w) for w, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(w * (L + 52))` thời gian; bộ nhớ `O(w * 52)`.

## 8. TRACE từng bước

### 8.1. Cách 3 cho ví dụ 1

In [13]:
def trace_sorted(words):
    seen = {}
    for w in words:
        even, odd = w[::2], w[1::2]
        key = "".join(sorted(even) + sorted(odd))
        status = "nhóm MỚI" if key not in seen else f"cùng nhóm với {seen[key]}"
        seen.setdefault(key, w)
        print(f"{w}: chẵn={even} lẻ={odd} -> sắp xếp {sorted(even)} + {sorted(odd)} -> khóa {key!r}  ({status})")
    return len(seen)

assert trace_sorted(examples[0][0]) == 3

abcd: chẵn=ac lẻ=bd -> sắp xếp ['a', 'c'] + ['b', 'd'] -> khóa 'acbd'  (nhóm MỚI)
cdab: chẵn=ca lẻ=db -> sắp xếp ['a', 'c'] + ['b', 'd'] -> khóa 'acbd'  (cùng nhóm với abcd)
cbad: chẵn=ca lẻ=bd -> sắp xếp ['a', 'c'] + ['b', 'd'] -> khóa 'acbd'  (cùng nhóm với abcd)
xyzz: chẵn=xz lẻ=yz -> sắp xếp ['x', 'z'] + ['y', 'z'] -> khóa 'xzyz'  (nhóm MỚI)
zzxy: chẵn=zx lẻ=zy -> sắp xếp ['x', 'z'] + ['y', 'z'] -> khóa 'xzyz'  (cùng nhóm với xyzz)
zzyx: chẵn=zy lẻ=zx -> sắp xếp ['y', 'z'] + ['x', 'z'] -> khóa 'yzxz'  (nhóm MỚI)


### 8.2. Cách 3 cho ví dụ 2 (`L = 3`)

Với `L = 3` chỉ số chẵn là `0, 2` (hoán đổi được) và chỉ số lẻ chỉ có `1`: ký tự giữa cố định. Khóa là `(hai đầu đã sắp xếp) + giữa`.

In [14]:
assert trace_sorted(examples[1][0]) == 3

abc: chẵn=ac lẻ=b -> sắp xếp ['a', 'c'] + ['b'] -> khóa 'acb'  (nhóm MỚI)
acb: chẵn=ab lẻ=c -> sắp xếp ['a', 'b'] + ['c'] -> khóa 'abc'  (nhóm MỚI)
bac: chẵn=bc lẻ=a -> sắp xếp ['b', 'c'] + ['a'] -> khóa 'bca'  (nhóm MỚI)
bca: chẵn=ba lẻ=c -> sắp xếp ['a', 'b'] + ['c'] -> khóa 'abc'  (cùng nhóm với acb)
cab: chẵn=cb lẻ=a -> sắp xếp ['b', 'c'] + ['a'] -> khóa 'bca'  (cùng nhóm với bac)
cba: chẵn=ca lẻ=b -> sắp xếp ['a', 'c'] + ['b'] -> khóa 'acb'  (cùng nhóm với abc)


### 8.3. Cách 1 (BFS) cho một chuỗi nhỏ

Quỹ đạo của `"abcd"`: chỉ số chẵn `0, 2` hoán đổi được (`a`, `c`), chỉ số lẻ `1, 3` hoán đổi được (`b`, `d`): `2 * 2 = 4` chuỗi.

In [15]:
o = orbit("abcd")
print(sorted(o))
assert o == {"abcd", "cbad", "adcb", "cdab"}
print("hai chuỗi trong quỹ đạo có cùng khóa:", {key_sorted(x) for x in o})

['abcd', 'adcb', 'cbad', 'cdab']
hai chuỗi trong quỹ đạo có cùng khóa: {'acbd'}


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (khóa là chuỗi đã sắp xếp theo chẵn/lẻ): một dòng, `O(w * L log L)`, rất dễ đúng.

```python
class Solution:
    def numSpecialEquivGroups(self, words: List[str]) -> int:
        return len({"".join(sorted(w[::2]) + sorted(w[1::2])) for w in words})
```

In [16]:
class Solution:
    def numSpecialEquivGroups(self, words: List[str]) -> int:
        return len({"".join(sorted(w[::2]) + sorted(w[1::2])) for w in words})

sol = Solution()
for words, expected in examples:
    got = sol.numSpecialEquivGroups(words)
    print(words, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['abcd', 'cdab', 'cbad', 'xyzz', 'zzxy', 'zzyx'] -> 3 OK
['abc', 'acb', 'bac', 'bca', 'cab', 'cba'] -> 3 OK


## 10. Vì sao đúng

**(a) Bất biến.** Một nước đi hoán đổi hai ký tự cùng chẵn (hoặc cùng lẻ), nên đa tập ký tự ở vị trí chẵn và đa tập ký tự ở vị trí lẻ **không đổi**. Vậy nếu `a` biến được thành `b` thì hai chuỗi có cùng đa tập chẵn và cùng đa tập lẻ.

**(b) Đủ.** Ngược lại, nếu `a`, `b` có cùng đa tập chẵn và cùng đa tập lẻ thì ta biến `a` thành `b` bằng **sắp xếp chọn (selection sort)** trong từng nhóm: duyệt chỉ số `i` từ trái sang phải; nếu `a[i] != b[i]`, tìm chỉ số `j > i` **cùng tính chẵn lẻ** có `a[j] == b[i]` (tồn tại vì đa tập các ký tự chưa cố định còn lại của nhóm đó của `a` và của `b` bằng nhau) rồi hoán đổi `a[i]` với `a[j]`. Mỗi bước hoán đổi cùng chẵn/lẻ nên hợp lệ; sau khi duyệt hết `a == b`.

**(c) Tương đương ⇒ khóa bằng nhau và ngược lại.** Các khóa của Cách 3 là đa tập chẵn đã sắp xếp nối đa tập lẻ đã sắp xếp. Do mọi chuỗi cùng độ dài `L`, độ dài phần chẵn là `ceil(L/2)` cố định, nên phép nối **không nhập nhằng**: từ khóa ta đọc lại đúng hai đa tập. Vậy khóa bằng nhau ⇔ hai đa tập bằng nhau ⇔ tương đương (theo (a), (b)). Cách 4 mã hóa cùng hai đa tập bằng bảng đếm.

**(d) Số nhóm = số khóa khác nhau.** Quan hệ tương đương chia `words` thành các lớp rời nhau; mỗi lớp là một nhóm lớn nhất (không thêm được từ ngoài lớp vì từ đó không tương đương với mọi thành viên). Số lớp bằng số khóa khác nhau.

**Kiểm chứng bằng máy.** Hai oracle **độc lập** với các lời giải: (1) **BFS quỹ đạo** theo đúng định nghĩa (`L <= 8`); (2) **thuật toán hoán đổi cấu trúc**: kiểm tra `a` biến được thành `b` bằng chính cách sắp xếp chọn ở (b), có **ghi lại dãy nước đi** và kiểm tra từng nước là hoán đổi cùng chẵn/lẻ (dùng được với `L` tới 20). Đếm nhóm của oracle (2) bằng so sánh với các đại diện.

In [17]:
def can_transform(a: str, b: str):
    """Thử biến a thành b bằng sắp xếp chọn trong từng nhóm chẵn/lẻ; trả về dãy nước đi (i, j) hoặc None."""
    if len(a) != len(b):
        return None
    cur = list(a)
    moves = []
    for i in range(len(cur)):
        if cur[i] == b[i]:
            continue
        j = next((j for j in range(i + 2, len(cur), 2) if cur[j] == b[i]), None)
        if j is None:
            return None
        cur[i], cur[j] = cur[j], cur[i]
        moves.append((i, j))
    return moves

def check_moves(a: str, b: str, moves) -> bool:
    cur = list(a)
    for i, j in moves:
        if i % 2 != j % 2 or i == j:
            return False
        cur[i], cur[j] = cur[j], cur[i]
    return "".join(cur) == b

def groups_oracle2(words: List[str]) -> int:
    reps: List[str] = []
    for w in words:
        found = False
        for r in reps:
            mv = can_transform(r, w)
            if mv is not None:
                assert check_moves(r, w, mv)           # bằng chứng: các nước đi hợp lệ
                found = True
                break
        if not found:
            reps.append(w)
    return len(reps)

rng = random.Random(893)
for _ in range(400):
    L = rng.randint(1, 7)
    alphabet = string.ascii_lowercase[:rng.randint(1, 4)]
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 8))]
    assert groups_brute(words) == groups_oracle2(words), words
print("oracle BFS quỹ đạo và oracle hoán đổi cấu trúc khớp nhau trên 400 danh sách ngẫu nhiên")

oracle BFS quỹ đạo và oracle hoán đổi cấu trúc khớp nhau trên 400 danh sách ngẫu nhiên


## 11. Time / Space Complexity

Gọi `w` là số từ, `L` độ dài mỗi từ, `g` là số nhóm.

| Cách | Time | Space |
|---|---|---|
| 1. BFS quỹ đạo | `O(w * (L/2)!^2 ...)` (giai thừa) | `O((L/2)!^2)` |
| 2. So với đại diện | `O(w * g * L log L)` | `O(g * L)` |
| 3. Khóa đã sắp xếp | `O(w * L log L)` | `O(w * L)` |
| 4. Khóa bảng đếm | `O(w * L)` | `O(w * 52)` |

Đo thử với `w = 1000`, `L = 20` (cỡ tối đa), dữ liệu có nhiều nhóm và nhiều chuỗi trùng khóa:

In [18]:
rng = random.Random(11)
words = []
bases = ["".join(rng.choice("abcdefgh") for _ in range(20)) for _ in range(150)]
for _ in range(1000):
    b = list(rng.choice(bases))
    for parity in (0, 1):
        idx = list(range(parity, 20, 2))
        vals = [b[i] for i in idx]
        rng.shuffle(vals)
        for i, v in zip(idx, vals):
            b[i] = v
    words.append("".join(b))
ref = groups_oracle2(words)
for name, fn in (("Cách 2 đại diện", sol_reps), ("Cách 3 khóa sắp xếp", sol_sorted), ("Cách 4 khóa đếm", sol_count)):
    t0 = time.perf_counter()
    got = fn(words)
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref, name
    print(f"{name:20} w=1000, L=20: {dt:8.2f} ms (số nhóm {got})")

Cách 2 đại diện      w=1000, L=20:    66.41 ms (số nhóm 150)
Cách 3 khóa sắp xếp  w=1000, L=20:     1.48 ms (số nhóm 150)
Cách 4 khóa đếm      w=1000, L=20:     1.86 ms (số nhóm 150)


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với hai oracle, test ngẫu nhiên có seed (đúng ràng buộc) và test có đáp án biết trước (sinh chuỗi bằng cách xáo trộn trong từng nhóm chẵn/lẻ).

In [19]:
def solutions_list():
    return {
        "reps": groups_reps,
        "sorted": groups_sorted,
        "count": groups_count,
        "sol_reps": sol_reps,
        "sol_sorted": sol_sorted,
        "sol_count": sol_count,
        "final": Solution().numSpecialEquivGroups,
    }

def run_all(words, expected=None, brute_limit=8):
    ref = groups_oracle2(words)
    if len(words[0]) <= brute_limit:
        assert groups_brute(words) == ref, ("oracle lệch", words)
    for name, fn in solutions_list().items():
        assert fn(list(words)) == ref, (name, words)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", words, expected, ref)
    return ref

# 12.1 ví dụ đề
for words, expected in examples:
    assert run_all(words, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (["a"], 1),                                  # một từ, một ký tự
    (["a", "a", "a"], 1),                        # trùng lặp: cùng nhóm
    (["a", "b", "c"], 3),                        # độ dài 1: không có nước đi nào
    (["ab", "ba"], 2),                           # độ dài 2: mỗi nhóm chỉ có một chỉ số, không hoán đổi được
    (["ab", "ab"], 1),
    (["abc", "cba"], 1),                         # chỉ số 0 và 2 (chẵn) hoán đổi được
    (["abc", "bac"], 2),                         # bac đổi vị trí ký tự chẵn/lẻ: khác nhóm
    (["aabb", "bbaa", "abab", "baba"], 3),       # {aabb,bbaa}: chẵn {a,b} lẻ {a,b}; abab; baba
    (["abcd", "cdab", "cbad", "xyzz", "zzxy", "zzyx"], 3),
    (["abcde", "edcba", "ebcda", "badce"], 2),   # abcde~edcba~ebcda (chẵn {a,c,e}, lẻ {b,d}); badce: chẵn {b,d,e}
    (["z" * 20, "z" * 20], 1),                   # độ dài 20 toàn giống nhau
    ([string.ascii_lowercase[:20], string.ascii_lowercase[:20][::-1]], 2),   # đảo ngược độ dài chẵn: chẵn <-> lẻ
]
for words, expected in edge:
    assert run_all(words, expected) == expected, words
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_anagram(words):                         # sắp xếp cả chuỗi (coi như anagram)
    return len({"".join(sorted(w)) for w in words})

def wrong_even_only(words):                       # chỉ xét nhóm chẵn
    return len({"".join(sorted(w[::2])) for w in words})

def wrong_odd_only(words):                        # chỉ xét nhóm lẻ
    return len({"".join(sorted(w[1::2])) for w in words})

def wrong_set_not_multiset(words):                # dùng set thay vì đa tập
    return len({(frozenset(w[::2]), frozenset(w[1::2])) for w in words})

def wrong_unordered_halves(words):                # coi (chẵn, lẻ) và (lẻ, chẵn) là như nhau
    return len({frozenset([("".join(sorted(w[::2]))), ("".join(sorted(w[1::2])))]) for w in words})

def find_counter(wrong, tries=30000, seed=5):
    rng = random.Random(seed)
    for _ in range(tries):
        L = rng.randint(2, 6)
        alphabet = string.ascii_lowercase[:rng.randint(2, 3)]
        words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(2, 5))]
        if wrong(words) != groups_oracle2(words):
            return words
    return None

counters = {}
for name, fn in (("coi như anagram", wrong_anagram), ("chỉ nhóm chẵn", wrong_even_only), ("chỉ nhóm lẻ", wrong_odd_only),
                 ("set thay cho đa tập", wrong_set_not_multiset), ("hoán đổi vai trò chẵn/lẻ", wrong_unordered_halves)):
    words = find_counter(fn)
    assert words is not None, name
    counters[name] = words
    print(f"{name:26} words={words}: sai = {fn(words)}, đúng = {groups_oracle2(words)}")
    run_all(words)
print("12.3 phản ví dụ: OK")

coi như anagram            words=['bbbb', 'abba', 'abab', 'babb']: sai = 3, đúng = 4
chỉ nhóm chẵn              words=['aac', 'cba', 'aaa']: sai = 2, đúng = 3
chỉ nhóm lẻ                words=['aac', 'cba', 'aaa']: sai = 2, đúng = 3
set thay cho đa tập        words=['aaaaa', 'bbbbb', 'abbab', 'aabba', 'bbabb']: sai = 4, đúng = 5
hoán đổi vai trò chẵn/lẻ   words=['ccbb', 'caca', 'bcbb', 'cbbb']: sai = 3, đúng = 4
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: so với oracle BFS quỹ đạo (L <= 8) và oracle hoán đổi
rng = random.Random(4893)
for _ in range(1200):
    L = rng.randint(1, 8)
    alphabet = string.ascii_lowercase[:rng.randint(1, 4)]
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 12))]
    run_all(words)
print("12.4 ngẫu nhiên nhỏ (1200 ca, gồm oracle BFS): OK")

# 12.5 sinh có đáp án biết trước: chọn các "đại diện" khác nhóm đôi một, rồi sinh biến thể bằng xáo trộn trong từng nhóm chẵn/lẻ
rng = random.Random(5893)
def shuffle_within_parity(rng, s):
    t = list(s)
    for parity in (0, 1):
        idx = list(range(parity, len(t), 2))
        vals = [t[i] for i in idx]
        rng.shuffle(vals)
        for i, v in zip(idx, vals):
            t[i] = v
    return "".join(t)

for _ in range(250):
    L = rng.randint(1, 20)
    alphabet = string.ascii_lowercase[:rng.randint(1, 26)]
    bases, keys = [], set()
    for _ in range(rng.randint(1, 12)):
        b = "".join(rng.choice(alphabet) for _ in range(L))
        k = key_sorted(b)
        if k not in keys:                         # bảo đảm các đại diện khác nhóm đôi một
            keys.add(k)
            bases.append(b)
    words = []
    for b in bases:
        for _ in range(rng.randint(1, 6)):
            words.append(shuffle_within_parity(rng, b))
    rng.shuffle(words)
    assert run_all(words, len(bases), brute_limit=0) == len(bases)
print("12.5 sinh có đáp án biết trước (250 ca, L tới 20): OK")

# 12.6 lớn: w = 1000, L = 20 nhiều dạng
rng = random.Random(6893)
shapes = {
    "toàn khác nhau": ["".join(rng.choice(string.ascii_lowercase) for _ in range(20)) for _ in range(1000)],
    "toàn giống nhau": ["abcdefghijklmnopqrst"] * 1000,
    "bảng chữ cái 2 ký tự": ["".join(rng.choice("ab") for _ in range(20)) for _ in range(1000)],
}
for name, words in shapes.items():
    ref = groups_oracle2(words)
    for fname, fn in solutions_list().items():
        assert fn(list(words)) == ref, (name, fname)
    print(f"  {name:24} số nhóm = {ref}")
print("12.6 w = 1000, L = 20: OK")

# 12.7 tính chất: xáo trộn thứ tự words không đổi đáp án; thêm một từ làm đáp án tăng 0 hoặc 1
rng = random.Random(7893)
for _ in range(300):
    L = rng.randint(1, 10)
    alphabet = string.ascii_lowercase[:rng.randint(1, 4)]
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 10))]
    base = Solution().numSpecialEquivGroups(words)
    shuffled = words[:]
    rng.shuffle(shuffled)
    assert Solution().numSpecialEquivGroups(shuffled) == base
    extra = "".join(rng.choice(alphabet) for _ in range(L))
    assert Solution().numSpecialEquivGroups(words + [extra]) - base in (0, 1)
print("12.7 tính chất: OK")

12.4 ngẫu nhiên nhỏ (1200 ca, gồm oracle BFS): OK


12.5 sinh có đáp án biết trước (250 ca, L tới 20): OK


  toàn khác nhau           số nhóm = 1000
  toàn giống nhau          số nhóm = 1
  bảng chữ cái 2 ký tự     số nhóm = 69
12.6 w = 1000, L = 20: OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi như anagram (sắp xếp cả chuỗi)

Sắp xếp **cả** chuỗi cho phép đưa ký tự từ vị trí chẵn sang lẻ, điều bị cấm.

In [22]:
words = counters["coi như anagram"]
print("words =", words, "| sai ->", wrong_anagram(words), "| đúng ->", Solution().numSpecialEquivGroups(words))
assert wrong_anagram(words) != Solution().numSpecialEquivGroups(words)
print("ví dụ 2:", wrong_anagram(examples[1][0]), "(sai: cả 6 hoán vị là anagram) so với", Solution().numSpecialEquivGroups(examples[1][0]))

words = ['bbbb', 'abba', 'abab', 'babb'] | sai -> 3 | đúng -> 4
ví dụ 2: 1 (sai: cả 6 hoán vị là anagram) so với 3


Vì sao sai: `"abc"` và `"bac"` là anagram nhưng không tương đương đặc biệt (ký tự `b` ở vị trí lẻ không thể sang chẵn). **Đúng:** sắp xếp **riêng** nhóm chẵn và nhóm lẻ.

### Lỗi 2 — Chỉ xét một nhóm (chẵn hoặc lẻ)

Nhóm còn lại cũng quyết định tính tương đương.

In [23]:
for name, fn in (("chỉ nhóm chẵn", wrong_even_only), ("chỉ nhóm lẻ", wrong_odd_only)):
    words = counters[name]
    print(f"{name}: words = {words} | sai -> {fn(words)} | đúng -> {Solution().numSpecialEquivGroups(words)}")
    assert fn(words) != Solution().numSpecialEquivGroups(words)

chỉ nhóm chẵn: words = ['aac', 'cba', 'aaa'] | sai -> 2 | đúng -> 3
chỉ nhóm lẻ: words = ['aac', 'cba', 'aaa'] | sai -> 2 | đúng -> 3


Vì sao sai: hai chuỗi có cùng nhóm chẵn nhưng nhóm lẻ khác nhau (hoặc ngược lại) không tương đương. **Đúng:** khóa gồm **cả hai** nhóm.

### Lỗi 3 — Dùng `set` thay cho đa tập

`set` bỏ qua số lần xuất hiện của mỗi ký tự.

In [24]:
words = counters["set thay cho đa tập"]
print("words =", words, "| sai ->", wrong_set_not_multiset(words), "| đúng ->", Solution().numSpecialEquivGroups(words))
assert wrong_set_not_multiset(words) != Solution().numSpecialEquivGroups(words)

words = ['aaaaa', 'bbbbb', 'abbab', 'aabba', 'bbabb'] | sai -> 4 | đúng -> 5


Vì sao sai: `{a, a, b}` và `{a, b, b}` có cùng `set` nhưng khác đa tập. **Đúng:** `sorted(...)` giữ lại số lần xuất hiện (hoặc bảng đếm).

### Lỗi 4 — Gộp (chẵn, lẻ) với (lẻ, chẵn)

Dùng khóa không có thứ tự như `frozenset({chẵn, lẻ})` làm hai chuỗi có vai trò chẵn/lẻ **bị đảo** trông như nhau.

In [25]:
words = counters["hoán đổi vai trò chẵn/lẻ"]
print("words =", words, "| sai ->", wrong_unordered_halves(words), "| đúng ->", Solution().numSpecialEquivGroups(words))
assert wrong_unordered_halves(words) != Solution().numSpecialEquivGroups(words)
print("ví dụ 1: zzxy / zzyx ->", wrong_unordered_halves(["zzxy", "zzyx"]), "(sai: gộp) so với", Solution().numSpecialEquivGroups(["zzxy", "zzyx"]))

words = ['ccbb', 'caca', 'bcbb', 'cbbb'] | sai -> 3 | đúng -> 4
ví dụ 1: zzxy / zzyx -> 1 (sai: gộp) so với 2


Vì sao sai: `"zzxy"` có (chẵn, lẻ) = (`xz`, `yz`) còn `"zzyx"` có (`yz`, `xz`): đảo vai trò, không tương đương (đề lưu ý rõ). **Đúng:** khóa có thứ tự: (đa tập chẵn, đa tập lẻ).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "các phép biến đổi cho phép sắp xếp lại tùy ý trong từng lớp" ⇒ bất biến là **đa tập của từng lớp**; dùng **dạng chuẩn** để đếm lớp tương đương.
- **Cách làm quen thuộc:** khóa băm = tuple/chuỗi đã sắp xếp hoặc bảng đếm; `len(set(keys))`.
- Cắt lát `s[::2]`, `s[1::2]` cho chẵn/lẻ; chú ý cùng độ dài nên phép nối khóa không nhập nhằng (hoặc dùng tuple hai phần).
- Cùng họ: Group Anagrams (49), Valid Anagram (242), Find and Replace Pattern (890), Count Number of Nice Subarrays (khác ý tưởng), Number of Distinct Islands (694).
- Test nên có: một từ, trùng lặp, độ dài 1 (không có nước đi), độ dài 2 (chẵn/lẻ chỉ một phần tử), đảo vai trò chẵn/lẻ, độ dài 20.

## 15. Biến thể

1. **Tổng quát hóa theo bước `k`:** cho phép hoán đổi hai ký tự có chỉ số **cùng số dư khi chia cho `k`**; khóa là `k` đa tập (bài tập mở rộng bên dưới).
2. **Số chuỗi trong một quỹ đạo** (bài tập 5): `(ceil(L/2)! / tích các giai thừa số đếm chẵn) * (floor(L/2)! / tích các giai thừa số đếm lẻ)`.
3. **Trả về các nhóm** (không chỉ số lượng): `dict` khóa -> danh sách.
4. Bài liên quan: Group Anagrams (49).

Dưới đây là biến thể 1, kiểm bằng BFS quỹ đạo tổng quát.

In [26]:
def groups_mod_k(words: List[str], k: int) -> int:
    return len({tuple("".join(sorted(w[r::k])) for r in range(k)) for w in words})

def orbit_mod_k(w: str, k: int) -> set:
    seen, stack = {w}, [w]
    while stack:
        s = stack.pop()
        for r in range(k):
            for i, j in combinations(range(r, len(s), k), 2):
                t = list(s)
                t[i], t[j] = t[j], t[i]
                t = "".join(t)
                if t not in seen:
                    seen.add(t)
                    stack.append(t)
    return seen

def groups_mod_k_brute(words: List[str], k: int) -> int:
    reps = []
    for w in words:
        if not any(w in o for o in reps):
            reps.append(orbit_mod_k(w, k))
    return len(reps)

rng = random.Random(15)
for _ in range(300):
    k = rng.randint(1, 4)
    L = rng.randint(1, 7)
    alphabet = string.ascii_lowercase[:rng.randint(1, 3)]
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 7))]
    assert groups_mod_k(words, k) == groups_mod_k_brute(words, k), (words, k)
    if k == 2:
        assert groups_mod_k(words, 2) == Solution().numSpecialEquivGroups(words)
print("k = 2 trùng bài gốc; k = 1 là anagram:", groups_mod_k(["abc", "bca", "xyz"], 1), "; k = 2 ví dụ 2:", groups_mod_k(examples[1][0], 2))

k = 2 trùng bài gốc; k = 1 là anagram: 2 ; k = 2 ví dụ 2: 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS quỹ đạo | Định nghĩa | giai thừa | giai thừa | Oracle, `L <= 8` |
| 2. So với đại diện | `sorted` chẵn/lẻ từng cặp | `O(w g L log L)` | `O(g L)` | Dễ hiểu |
| 3. Khóa sắp xếp | `sorted(chẵn) + sorted(lẻ)` | `O(w L log L)` | `O(w L)` | **Cách nộp chính** |
| 4. Khóa bảng đếm | 52 ô đếm | `O(w L)` | `O(w)` | Tránh sắp xếp, `L` lớn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
w = "zzxy"
print(w[::2], w[1::2])
print(key_sorted("zzxy"), key_sorted("xyzz"))
print(key_sorted("zzxy") == key_sorted("zzyx"))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `words = ["aabb", "abab", "baba", "bbaa"]`: ghi số nhóm và các nhóm (dưới dạng danh sách các tập từ).

In [28]:
# Bài 2: điền kết quả tính tay
hand_count = None
hand_groups = None            # ví dụ [{"..", ".."}, {".."}]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành khóa đã sắp xếp.

In [29]:
def exercise_3_key(w: str) -> str:
    # TODO 1: lấy các ký tự ở chỉ số chẵn và sắp xếp
    # TODO 2: lấy các ký tự ở chỉ số lẻ và sắp xếp
    # TODO 3: nối hai phần thành một chuỗi
    pass


# assert exercise_3_key("zzxy") == exercise_3_key("xyzz")
# assert exercise_3_key("zzxy") != exercise_3_key("zzyx")
# assert len({exercise_3_key(w) for w in examples[0][0]}) == 3

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_count(words)` với khóa là bảng đếm 52 phần tử.

In [30]:
def exercise_4_count(words: List[str]) -> int:
    # TODO
    pass


# assert exercise_4_count(examples[0][0]) == 3
# assert exercise_4_count(examples[1][0]) == 3

### Bài 5 — Mini challenge: số chuỗi trong một nhóm (khó)

Với một chuỗi `w`, hãy viết `exercise_5_orbit_size(w)` trả về **số chuỗi khác nhau** có thể thu được từ `w` bằng các nước đi (kể cả `w`), mà **không** liệt kê chúng.

```python
exercise_5_orbit_size("abcd")   -> 4     # 2! * 2!
exercise_5_orbit_size("aaaa")   -> 1
exercise_5_orbit_size("aabb")   -> 4     # chẵn {a,b}: 2 cách; lẻ {a,b}: 2 cách
exercise_5_orbit_size("abc")    -> 2     # chẵn {a,c}: 2 cách; lẻ {b}: 1 cách
```
Gợi ý: đếm số hoán vị khác nhau của một đa tập: `n! / (c1! * c2! * ...)`; nhân kết quả của nhóm chẵn và nhóm lẻ.

In [31]:
def exercise_5_orbit_size(w: str) -> int:
    # TODO
    pass


# assert exercise_5_orbit_size("abcd") == 4
# assert exercise_5_orbit_size("aaaa") == 1
# assert exercise_5_orbit_size("aabb") == 4
# assert exercise_5_orbit_size("abc") == 2

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `w[::2]` là chỉ số chẵn, `w[1::2]` là chỉ số lẻ.
- Mức 2: `"zzxy"` chẵn `zx`, lẻ `zy`; sắp xếp thành `xz` và `yz`.
- Mức 3: `zx zy`, `xzyz xzyz`, `False`.

**Bài 2**
- Mức 1: tính (chẵn đã sắp xếp, lẻ đã sắp xếp) cho từng từ.
- Mức 2: `aabb`: (`ab`,`ab`); `abab`: (`aa`,`bb`); `baba`: (`bb`,`aa`); `bbaa`: (`ab`,`ab`).
- Mức 3: `3` nhóm: `{aabb, bbaa}`, `{abab}`, `{baba}`.

**Bài 3**
- Mức 1: `sorted(w[::2])` là danh sách; `sorted(w[1::2])` là danh sách.
- Mức 2: nối hai danh sách bằng `+`.
- Mức 3: `return "".join(sorted(w[::2]) + sorted(w[1::2]))`.

**Bài 4**
- Mức 1: `cnt = [0] * 52` cho mỗi từ.
- Mức 2: `cnt[(i & 1) * 26 + ord(ch) - 97] += 1`.
- Mức 3: thêm `tuple(cnt)` vào `set`; trả `len(set)`.

**Bài 5**
- Mức 1: nhóm chẵn và nhóm lẻ độc lập nhau.
- Mức 2: số hoán vị của đa tập: `n! / (c1! * c2! * ...)`.
- Mức 3: nhân số hoán vị của nhóm chẵn với số hoán vị của nhóm lẻ.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
w = "zzxy"
print(w[::2], w[1::2])
print(key_sorted("zzxy"), key_sorted("xyzz"))
print(key_sorted("zzxy") == key_sorted("zzyx"))

assert (w[::2], w[1::2]) == ("zx", "zy")
assert (key_sorted("zzxy"), key_sorted("xyzz")) == ("xzyz", "xzyz")
assert (key_sorted("zzxy") == key_sorted("zzyx")) is False

zx zy
xzyz xzyz
False


### Đáp án Bài 2

Khóa: `aabb` -> (`ab`,`ab`), `abab` -> (`aa`,`bb`), `baba` -> (`bb`,`aa`), `bbaa` -> (`ab`,`ab`): ba nhóm.

In [33]:
words = ["aabb", "abab", "baba", "bbaa"]
hand_count_solution = 3
hand_groups_solution = [{"aabb", "bbaa"}, {"abab"}, {"baba"}]
assert Solution().numSpecialEquivGroups(words) == hand_count_solution == groups_brute(words)
# kiểm tra nhóm: mọi cặp trong cùng nhóm tương đương, khác nhóm thì không
for g in hand_groups_solution:
    assert all(w in orbit(r) for r in g for w in g)
for g1, g2 in combinations(hand_groups_solution, 2):
    assert all(w not in orbit(r) for r in g1 for w in g2)
assert sorted(w for g in hand_groups_solution for w in g) == sorted(words)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_key_solution(w: str) -> str:
    return "".join(sorted(w[::2]) + sorted(w[1::2]))

assert exercise_3_key_solution("zzxy") == exercise_3_key_solution("xyzz")
assert exercise_3_key_solution("zzxy") != exercise_3_key_solution("zzyx")
assert len({exercise_3_key_solution(w) for w in examples[0][0]}) == 3
rng = random.Random(3)
for _ in range(300):
    L = rng.randint(1, 8)
    alphabet = string.ascii_lowercase[:rng.randint(1, 3)]
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 8))]
    assert len({exercise_3_key_solution(w) for w in words}) == groups_brute(words)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_count_solution(words: List[str]) -> int:
    seen = set()
    for w in words:
        cnt = [0] * 52
        for i, ch in enumerate(w):
            cnt[(i & 1) * 26 + ord(ch) - 97] += 1
        seen.add(tuple(cnt))
    return len(seen)

assert exercise_4_count_solution(examples[0][0]) == 3
assert exercise_4_count_solution(examples[1][0]) == 3
rng = random.Random(4)
for _ in range(300):
    L = rng.randint(1, 8)
    alphabet = string.ascii_lowercase[:rng.randint(1, 3)]
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 8))]
    assert exercise_4_count_solution(words) == groups_brute(words)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_orbit_size_solution(w: str) -> int:
    def perms(part: str) -> int:
        total = factorial(len(part))
        for c in Counter(part).values():
            total //= factorial(c)
        return total
    return perms(w[::2]) * perms(w[1::2])

assert exercise_5_orbit_size_solution("abcd") == 4
assert exercise_5_orbit_size_solution("aaaa") == 1
assert exercise_5_orbit_size_solution("aabb") == 4
assert exercise_5_orbit_size_solution("abc") == 2
rng = random.Random(5)
for _ in range(300):
    L = rng.randint(1, 8)
    alphabet = string.ascii_lowercase[:rng.randint(1, 4)]
    w = "".join(rng.choice(alphabet) for _ in range(L))
    assert exercise_5_orbit_size_solution(w) == len(orbit(w)), w
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (khóa là chuỗi đã sắp xếp theo chẵn/lẻ)

```python
class Solution:
    def numSpecialEquivGroups(self, words: List[str]) -> int:
        # mỗi chuỗi -> (đa tập chẵn đã sắp xếp) + (đa tập lẻ đã sắp xếp); số khóa khác nhau = số nhóm
        return len({"".join(sorted(w[::2]) + sorted(w[1::2])) for w in words})
```

Công thức nhớ nhanh: tương đương ⇔ cùng `sorted(w[::2])` **và** cùng `sorted(w[1::2])`; số nhóm = số khóa khác nhau.

### Checklist trước khi submit

- [ ] Sắp xếp **riêng** nhóm chẵn và nhóm lẻ (không sắp xếp cả chuỗi)?
- [ ] Giữ **cả hai** nhóm trong khóa và theo **đúng thứ tự** (chẵn rồi lẻ)?
- [ ] Dùng đa tập (`sorted`/đếm), không dùng `set`?
- [ ] Mọi chuỗi cùng độ dài nên phép nối khóa không nhập nhằng (nếu khác độ dài hãy dùng tuple)?
- [ ] Đã thử: một từ, trùng lặp, độ dài 1 và 2, đảo vai trò chẵn/lẻ, độ dài 20?
- [ ] Tên method đúng `numSpecialEquivGroups(self, words)` và trả `int`?